In [38]:
# [نفس الكونسبت] — الـ imports، زي اللي عندك بالظبط + إضافتين احترافيتين:
# - MemorySaver: عشان الـ "ميموري" (حفظ حالة الجراف بين تشغيلة وتانية)
# - operator/Annotated: مش هنستخدمهم فعلياً هنا لكن لو حبيت بعدين تضيف "قائمة رسائل متراكمة" هتحتاجهم

from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np

from langchain_groq import ChatGroq
from langchain_core.messages import ToolMessage
from langchain_experimental.tools import PythonAstREPLTool

from typing import TypedDict, Optional
from langgraph.graph import START, END, StateGraph

from pydantic import BaseModel


load_dotenv()
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage
from langchain_community.tools import TavilySearchResults

In [39]:
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    api_key=os.getenv("groq"),
)


In [40]:
from pydantic import BaseModel, Field

class DataUnderstandingOutput(BaseModel):
    summary: str = Field(description="Clear data understanding summary for downstream agents (cleaning, analysis, etc.)")
    refined_question: str = Field(description="The refined and precise version of the user question with added context.")

In [41]:
from typing import TypedDict, Optional, Any

class agent_state(TypedDict):
    question: Optional[str]
    refined_question: Optional[str]
    csv_path: Optional[str]
    last_node_name: Optional[str]
    last_node_prompt: Optional[str]
    understand_node: Optional[Any]
    cleaner_node: Optional[Any]
    analyst_node: Optional[Any]
    decision_node: Optional[Any]
    raw_data: Optional[Any]
    clean_data: Optional[Any]
    new_prompt: Optional[str]
    prompt: Optional[str]
    supervisor_approved: Optional[bool]
    supervisor_feedback: Optional[str]
    supervisor_retry_count: Optional[int]

In [42]:
DATA_CACHE={}

In [43]:
def exctract_between_markers(text:str,marker_name:str):
    strat_marker=f"==={marker_name}_start==="
    end_marker=f"==={marker_name}_end==="
    if strat_marker in text and end_marker in text:
        return text.split(strat_marker)[1].split(end_marker)[0]
    else: print("no markers ")
        

In [44]:
def load_data_node(state: agent_state, config):
    thread_id = config["configurable"]["thread_id"]
    df = pd.read_csv(state.get("csv_path", "your_data.csv"))
    DATA_CACHE[thread_id] = {"raw": df}
    return {"last_node_name": "load_data_node"}


In [45]:
from langchain_core.tools import tool
import sys
import io

def make_python_tool(namespace: dict):
    """
    بترجع أداة بايثون مربوطة بـ namespace خاص بيها (dict فيه df + أي مكتبات).
    ليه namespace بدل الـ globals() الحقيقية بتاعة النوت بوك؟
    عشان لما الـ LLM يعمل مثلاً df_clean = df.dropna() جوه الكود اللي بيبعته،
    المتغير الجديد ده يتحفظ جوه نفس الـ dict ده، ونقدر نرجعه تاني بعد ما
    الـ agent يخلص شغله (شوف run_agent_with_tools تحت). ده اللي كان ناقص
    قبل كده وكان بيخلي أي تنظيف بيعمله الـ LLM يضيع.
    """
    @tool
    def python_repl_ast(code: str) -> str:
        """A Python shell. Use this to execute python code to inspect or clean pandas DataFrames.
        You have access to a pandas DataFrame named `df`. Always print the output to see results."""
        old_stdout = sys.stdout
        redirected_output = sys.stdout = io.StringIO()
        try:
            exec(code, namespace, namespace)
            sys.stdout = old_stdout
            return redirected_output.getvalue()
        except Exception as e:
            sys.stdout = old_stdout
            return f"Error: {e}"
    return python_repl_ast


In [ ]:
from pydantic import BaseModel, Field
from typing import List, Callable, Optional
def validation_run_agent(
    df,
    prompt: str,
    validate_fn: Callable[[dict], List[str]],   # بيرجع لستة مشاكل حتمية (فاضية = تمام)
    llm_validator: Optional[BaseModel] = None,   # Pydantic model لو عايز فحص دلالي كمان
    llm_validator_prompt: Optional[str] = None,
    extra_tools: list = None,
    max_iteration: int = 2,
    max_validation_retries: int = 2,
    name_space






        
):

In [ ]:
def my_task():
    tools=[python_repl_ast]
    pormpt="how to be a ai eng it exprmental prompt"
    messages = [HumanMessage(content=pormpt)]
    llm_with_tools=llm.bind_tools(tools)
    response= llm_with_tools.invoke(messages)
    while getattr(response,"tool_calls",None):
        messages.append(response)
        for tool_calls in response.tool_calls:
            result=python_repl_ast.invoke(tool_calls)
            messages.append(ToolMessage(content=str(result),tool_call_id=tool_calls["id"]))
        response= llm_with_tools.invoke(messages)
    return response

In [ ]:
def my_task():
    
    tools=[PythonAstREPLTool]
    llm_with_tools=llm.bind_tools(tools)
    prompt="you are exprmiant agent "
    messages.append(HumanMessage(content=prompt))
    response=llm_with_tools.invoke(messages)
    while getattr(response,"tool_calls"):
        messages.append(response)
        for tool_call in response.tool_calls:
            result=PythonAstREPLTool.invoke(tool_call["args"])
            messages.append(ToolMessage(content=result,tool_call_id=tool_call["id"]))
        response=llm_with_tools.invoke(messages)
    return response


In [ ]:
from pydantic import BaseModel, Field
from typing import List, Callable, Optional

def run_agent_with_validation(
    df,
    prompt: str,
    validate_fn: Callable[[dict], List[str]],   # بيرجع لستة مشاكل حتمية (فاضية = تمام)
    llm_validator: Optional[BaseModel] = None,   # Pydantic model لو عايز فحص دلالي كمان
    llm_validator_prompt: Optional[str] = None,
    extra_tools: list = None,
    max_iteration: int = 8,
    max_validation_retries: int = 3,
):
    """
    زي run_agent_with_tools بالظبط + خطوتين فحص بعد ما الـ LLM يقول 'خلصت':
    1) فحص حتمي بايثون (validate_fn) على namespace الحقيقي
    2) فحص دلالي بالـ LLM (اختياري) لو الحاجة محتاجة فهم مش مجرد أرقام

    لو في مشكلة، مش هنعيد النود - هنضيف رسالة فيها المشكلة بالظبط
    ونخلي نفس المحادثة تكمل عشان الـ LLM يصلح بس اللي غلط.
    """
    namespace = {"df": df, "pd": pd, "np": np}
    python_tool = make_python_tool(namespace)
    tools = [python_tool] + (extra_tools or [])
    llm_with_tools = llm.bind_tools(tools)
    tools_map = {t.name: t for t in tools}

    messages = [HumanMessage(content=prompt)]
    validation_log = []
    response = None

    for attempt in range(max_validation_retries + 1):
        response = llm_with_tools.invoke(messages)
        iteration = 0
        while getattr(response, "tool_calls", None) and iteration < max_iteration:
            messages.append(response)
            for tool_call in response.tool_calls:
                tool_name, tool_args = tool_call["name"], tool_call["args"]
                if tool_name in tools_map:
                    try:
                        result = tools_map[tool_name].invoke(tool_args)
                    except Exception as e:
                        result = f"Error executing tool {tool_name}: {e}"
                else:
                    result = f"Error: Tool '{tool_name}' not found."
                messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))
            response = llm_with_tools.invoke(messages)
            iteration += 1
        messages.append(response)

        # --- 1) الفحص الحتمي ---
        issues = validate_fn(namespace)

        # --- 2) الفحص الدلالي (اختياري) لو الحتمي عدّى ---
        if not issues and llm_validator and llm_validator_prompt:
            structured_llm = llm.with_structured_output(llm_validator)
            check = structured_llm.invoke(llm_validator_prompt.format(
                summary=namespace.get("df_clean", namespace.get("df")).describe(include="all").to_string()
            ))
            if not check.passed:
                issues = check.issues

        validation_log.append({"attempt": attempt, "issues": issues})
        if not issues:
            break
        if attempt == max_validation_retries:
            break  # وصلنا لأقصى محاولات - هنمرر الحالة زي ما هي مع علم فشل الفاليديشن

        feedback = "لقيت المشاكل دي في شغلك، صلحها من غير ما تعيد اللي شغال صح:\n" + \
                   "\n".join(f"- {i}" for i in issues)
        messages.append(HumanMessage(content=feedback))

    return response, namespace, validation_log

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

def run_agent_with_tools(df, prompt: str, extra_tools: list = None, max_iteration: int = 2):
    # namespace خاص بكل تشغيلة - فيه df + pandas/numpy عشان أي كود يكتبه الـ LLM يلاقيهم جاهزين
    namespace = {"df": df, "pd": pd, "np": np}
    python_tool = make_python_tool(namespace)

    # تجميع كل الأدوات في لستة واحدة (بايثون + أي أدوات إضافية زي السيرش)
    tools = [python_tool]
    if extra_tools:
        tools.extend(extra_tools)

    # عمل ربط للأدوات بالـ LLM
    llm_with_tools = llm.bind_tools(tools)

    # إنشاء دكشنري للوصول السريع للأداة عن طريق اسمها وقت الـ Loop
    tools_map = {tool.name: tool for tool in tools}

    messages = [HumanMessage(content=prompt)]
    response = llm_with_tools.invoke(messages)
    iteration = 0

    while getattr(response, "tool_calls", None) and iteration < max_iteration:
        messages.append(response)

        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            # التأكد إن الأداة موجودة وتنفيذها بناءً على اسمها
            if tool_name in tools_map:
                try:
                    result = tools_map[tool_name].invoke(tool_args)
                except Exception as e:
                    result = f"Error executing tool {tool_name}: {e}"
            else:
                result = f"Error: Tool '{tool_name}' not found."

            messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

        response = llm_with_tools.invoke(messages)
        iteration += 1

    if iteration >= max_iteration:
        response.content = (response.content or "") + f"\n[WARNING: hit max_iterations ({max_iteration})]"

    # بنرجع الـ namespace كله (فيه df الأصلي + أي متغير جديد عمله الـ LLM زي df_clean)
    # بدل ما نرجع {"df": df} ثابتة كانت بتخلي أي تنظيف يضيع
    return response, namespace


In [47]:
def understand_node(state: agent_state, config):
    thread_id = config["configurable"]["thread_id"]
    df = DATA_CACHE[thread_id]["raw"]
    question = state.get("question", "")
    
    exploration_prompt = f"""
    You are an expert Senior Data Analyst. You have access to a pandas DataFrame named `df`.
    Use your Python tool to inspect `df` (e.g., info, missing values, head) to fully understand the data regarding this user question: "{question}".
    """
    
    # استدعاء الدالة واستقبال الـ response والـ variables
    response, _ = run_agent_with_tools(df, exploration_prompt)

    # استخدام الـ Structured Output للحصول على ملخص وسؤال دقيق
    structured_llm = llm.with_structured_output(DataUnderstandingOutput)

    final_prompt = f"""
    Based on the previous data exploration and the original user question: "{question}", 
    generate the final data understanding summary and the refined question.
    Previous findings: {response.content}
    """
    
    structured_result = structured_llm.invoke(final_prompt)
    
    # إرجاع البيانات وتحديث الـ State بالأسماء الصحيحة اللي بيدور عليها السوبرفايزر
    return {
        "last_node_name": "understand_node",
        "last_node_prompt": final_prompt,          # مهم جداً عشان السوبرفايزر يشوفه
        "understand_node": structured_result.summary,
        "refined_question": structured_result.refined_question,
        "supervisor_retry_count": 0                # تصفير العداد عند النجاح
    }

run fn el llm with tools invoke llm -> tool -> llm , none ? 

In [48]:
def cleaner(state: agent_state, config):
    thread_id = config["configurable"]["thread_id"]
    df = DATA_CACHE[thread_id]["raw"]
    refined_question = state.get("refined_question", "")
    
    cleaning_prompt = f"""
    You are an elite Senior Data Cleaner and Quality Assurance Engineer. 
    You have access to a pandas DataFrame named `df`.
    The primary analytical goal is: "{refined_question}".

    Your task is to perform a **DEEP and COMPREHENSIVE cleaning** process. You must:
    1. **Analyze Anomalies & Outliers:** Detect numerical and logical anomalies.
    2. **Handle Missing Values Deeply:** Use smart imputation if dropping loses data.
    3. **Consistency & Formatting:** Fix string inconsistencies and standardize formats.
    4. **Save Output:** Save the final pristine DataFrame in a variable named `df_clean`.
    """
    
    cleaning_result, local_var = run_agent_with_tools(df, cleaning_prompt, max_iteration=10)
    cleaned_df = local_var.get("df_clean", df)
    
    # حفظ الداتا النضيفة في الـ Cache للـ Nodes اللي بعده
    DATA_CACHE[thread_id]["clean"] = cleaned_df
    
    return {
        "last_node_name": "cleaner_node",
        "last_node_prompt": cleaning_prompt,       # مهم جداً للـ Supervisor
        "cleaner_node": cleaning_result.content,
        "supervisor_retry_count": 0
    }

In [49]:
def analyst(state: agent_state, config):
    thread_id = config["configurable"]["thread_id"]
    df_clean = DATA_CACHE[thread_id]["clean"]
    # كان في الكود القديم state["understand"] وده مفيش زي ده في الـ state أصلاً
    # الاسم الصح المتخزن في الـ state هو "understand_node"
    understand = state.get("understand_node", "")

    analyst_prompt = f"""
You are an expert Senior Data Analyst. You have access to a cleaned pandas DataFrame named `df` (which is actually the pristine cleaned data).
The user's core analytical objective is: "{state.get('refined_question', '')}".
Data Understanding Context from previous steps:
"{understand}"

Your task is to perform a **deep and rigorous data analysis** to answer the user's question completely:
1. **Explore & Aggregate:** Use Python code (pandas, numpy, etc.) to calculate precise metrics, groupings, distributions, or trends required by the question.
2. **Answer with Evidence:** Do not guess. Base all your findings strictly on the numbers and results produced by your Python code execution.
3. **Print Key Insights:** Make sure to print out or summarize the key numerical findings clearly in your response so downstream agents or the final user can read them easily.
4. **Final Output:** Provide a comprehensive, data-driven answer that fully addresses the refined question.
"""
    # كان في الكود القديم: analyst_result = run_agent_with_tools(...)
    # المشكلة: الدالة بترجع tuple (response, locals) مش object واحد بس
    # فلما بعدين بيتعمل analyst_result.content كان هيطلع AttributeError
    analyst_result, _ = run_agent_with_tools(df_clean, analyst_prompt, max_iteration=12)

    return {
        "last_node_name": "analyst_node",
        "analyst_node": analyst_result.content,
        "prompt": analyst_prompt,
    }


In [50]:
from langchain_community.tools import TavilySearchResults

def decision_node(state: agent_state, config):
    thread_id = config["configurable"]["thread_id"]

    # بنسحب الداتا النضيفة
    df = DATA_CACHE[thread_id].get("clean", DATA_CACHE[thread_id]["raw"])

    # تجهيز أداة السيرش (محتاجة TAVILY_API_KEY في الـ .env عشان تشتغل)
    search_tool = TavilySearchResults(max_results=3)

    # البرومت الاستراتيجي
    # كان في الكود القديم state.get('understand', '') و state.get('analysis_result', '')
    # وده مفيش زيهم في الـ state، الأسماء الصح هي "understand_node" و "analyst_node"
    decision_prompt = f"""
    You are an elite Strategic Business Advisor.
    You have access to:
    - Original Question/Objective: "{state.get('refined_question', '')}"
    - Data Understanding: "{state.get('understand_node', '')}"
    - Analysis Results: "{state.get('analyst_node', '')}"

    Your task:
    1. Review the internal analysis.
    2. If you need external market benchmarks, industry standards, or validation, **use your search tool**.
    3. Provide the final strategic decision, key insights, justification, and alternative actions.
    """

    # استدعاء الدالة وباصي أداة السيرش في الـ extra_tools
    response, _ = run_agent_with_tools(
        df=df,
        prompt=decision_prompt,
        extra_tools=[search_tool],
        max_iteration=6
    )

    return {
        "last_node_name": "decision_node",
        "decision_node": response.content
    }


In [51]:
from pydantic import BaseModel, Field

class SupervisorDecision(BaseModel):
    approved: bool = Field(description="Set to True if the output meets the requirements and analysis goals, otherwise False.")
    feedback: str = Field(description="If approved, give a brief positive note. If rejected, provide clear, strict instructions and prompt adjustments for the agent to fix its mistakes.")

In [ ]:
def supervisor(state:agent_state,config):
    last_node=state["last_node_name"]
    last_node_prompt=state["last_node_prompt"]
    last_node_output=state[last_node]
    revision_count = state.get("supervisor_revision_count", 0)
    supervisor_prompt = f"""
    You are an elite AI Quality Assurance Supervisor. 
    Your job is to review the output generated by the previous agent (`{last_node}`).
    
    - **Original Prompt given to the agent:** 
      {last_node_prompt}
      
    - **Agent's Output:** 
      {last_node_output}
      
    - **Current Retry Count:** {revision_count} / 3
    
    Evaluate the output critically:
    1. Did the agent fully accomplish what was asked in the prompt?
    2. Are there any errors, missing information, or hallucinated facts?
    3. If retry_count >= 3, you MUST approve it (approved: True) to prevent infinite loops, even if it's not 100% perfect.
    
    Provide your decision (approved: True/False) and constructive feedback/corrections if rejected.
    """
    structured_supervisor=llm.with_structured_output(SupervisorDecision)
    decision = structured_supervisor.invoke(supervisor_prompt)
    new_revision_count = revision_count + 1 if not decision.approved else 0
    
    return {
        "supervisor_approved": decision.approved,
        "supervisor_feedback": decision.feedback,
        "supervisor_revision_count": newrevision_count
    }

In [53]:
def route_after_supervisor(state:agent_state):
    approved=state.get("supervisor_approved", True)
    retry_count=state.get("supervisor_retry_count", 0)
    last_node = state.get("last_node_name", "")
    if approved or retry_count > 3:
        if last_node == "understand_node":
            return "to_cleaner"
        elif last_node == "cleaner_node":
            return "to_analyst"
        elif last_node == "analyst_node":
            return "to_decision"
        elif last_node == "decision_node":
            return "end"
    else:
        # لو مرفوض، بنرجع لنفس النود تاني عشان تعيد الشغل بالـ feedback
        if last_node == "understand_node":
            return "retry_understand"
        elif last_node == "cleaner_node":
            return "retry_cleaner"
        elif last_node == "analyst_node":
            return "retry_analyst"
        elif last_node == "decision_node":
            return "retry_decision"
            
    return "end"


In [54]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

graph = StateGraph(agent_state)

# 1. الترتيب الصح: الاسم النصي أولاً ثم اسم الدالة
graph.add_node("load_data_node", load_data_node)
graph.add_node("understand_node", understand_node)
graph.add_node("supervisor", supervisor)
graph.add_node("decision_node", decision_node)
graph.add_node("analyst", analyst)
graph.add_node("cleaner", cleaner)

# 2. توصيل البداية
graph.add_edge(START, "load_data_node")
graph.add_edge("load_data_node", "understand_node")

# (اختياري لو الـ understand كمان بتروح للسوبرفايزر في الأول)
graph.add_edge("understand_node", "supervisor")

# 3. الـ Conditional Edges الخاصة بالسوبرفايزر
graph.add_conditional_edges(
    "supervisor",
    route_after_supervisor,
    {
        "to_cleaner": "cleaner",
        "retry_understand": "understand_node",
        
        "to_analyst": "analyst",
        "retry_cleaner": "cleaner",
        
        "to_decision": "decision_node",
        "retry_analyst": "analyst",
        
        "end": END,
        "retry_decision": "decision_node"
    }
)

# 4. ربط باقي النودز بالسوبرفايزر عشان يراجع وراهم
graph.add_edge("cleaner", "supervisor")
graph.add_edge("analyst", "supervisor")
graph.add_edge("decision_node", "supervisor")

# 5. عمل Compile للجراف
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

print("Graph compiled successfully with Supervisor!")

Graph compiled successfully with Supervisor!


In [55]:
# ملحوظة: عدّل مسار الملف ده على المسار الصحيح عندك قبل التشغيل
# (المسار القديم كان مسار خاص بجهاز معيّن على ويندوز وهيطلع FileNotFoundError عند أي حد تاني)

config = {"configurable": {"thread_id": "thread_1"}}
initial_state = {
    "question": "What are the main sales trends and how can we improve them?",
    "csv_path": r"D:\Downloads\data_analysis_agent\ecommerce_sales_analytics_5000.csv"  # <-- غيّر ده
}

# تشغيل الـ Pipeline (app اتعمله compile فوق في الخلية اللي فاتت، مش محتاجين نعمله تاني هنا)
for event in app.stream(initial_state, config):
    print(event)


{'load_data_node': {'last_node_name': 'load_data_node'}}


{'understand_node': {'last_node_name': 'understand_node', 'last_node_prompt': '\n    Based on the previous data exploration and the original user question: "What are the main sales trends and how can we improve them?", \n    generate the final data understanding summary and the refined question.\n    Previous findings: Based on my analysis of the 5,000-order dataset (Jan 2022 – Sep 2035, 4 product categories, 4 regions), here are the main sales trends and improvement recommendations.\n\n## Main Sales Trends\n\n**1. Overall revenue is flat, with no clear growth**\n- Annual revenue stays in a narrow ~$350K–$410K band across all years (e.g., 2022: $371.8K → 2035: $238.7K partial year). There is **no sustained upward trend** — sales are essentially flat/volatile.\n- Mild seasonality: May is the strongest month (~$1,105 avg order value), while April and January are the weakest (~$950–980).\n\n**2. Category mix**\n- **Electronics** is the top revenue driver (~$1.83M, ~34% of total), followed

<string>:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
<string>:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
<string>:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `qwen/qwen3.8-27b` in organization `org_01m23a9e0remht294xtzcbztmm` service tier `on_demand` on input tokens per minute (ITPM): Limit 7000, Requested 7094, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
from IPython.display import Image, display

try:
    # كان في الكود القديم متغير اسمه workflow ومش موجود أصلاً — المتغير الصح هو graph / app
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    # لو حصل مشكلة في تثبيت أدوات الرسم، بيطلع لك كود الـ Mermaid تقدر تعمله كوبي وتشوفه في أي Mermaid Live Editor
    print(app.get_graph().draw_mermaid())
